<a href="https://colab.research.google.com/github/Decoding-Data-Science/CommunityWorkshops/blob/main/bootcampsep26/Day_2_rag_llamaindex_sep26_pdf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install llama_index

In [ ]:
pip install llama-index-readers-file

In [4]:
import openai
from google.colab import userdata

# Retrieve the OpenAI API key from Google Colab secrets
openai.api_key = userdata.get('openai')

In [5]:
from llama_index.core import SimpleDirectoryReader
from llama_index.readers.file import PDFReader

documents = SimpleDirectoryReader(
    input_dir="data",
    required_exts=[".pdf"],
    file_extractor={".pdf": PDFReader()}
).load_data()

print(len(documents))
print(documents[0].text[:1000])

13
DDS Employee Handbook (Synthetic) v1
Effective date: March 03, 2026  Dubai (GST)
Note: This document is a synthetic, training-friendly employee handbook for demos, onboarding
simulations, and HR-policy chatbot prototypes. It is not legal advice and must be reviewed by qualified
counsel before any real-world use.
1. Welcome to Decoding Data Science (DDS)
DDS is a Dubai-based academy, consulting practice, and community focused on data science, AI, and
applied generative AI. We operate with a global mindset and a high trust culture—shipping practical
outcomes while supporting each other.
This handbook explains workplace expectations, benefits, and policies. If any local law conflicts with
this handbook, applicable law prevails.
2. Company Values & Ways of Working
 Build with clarity: define the user, problem, inputs/outputs, and definition of done.
 Bias for action: ship small, iterate fast, measure outcomes.
 Respect and inclusion: disagreement is allowed; disrespect is not.
 Dat

In [6]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
index = VectorStoreIndex.from_documents(documents=documents)
query_engine = index.as_query_engine()
response = query_engine.query("how many annual leave does does decoding data science have?")
print(response)

Decoding Data Science provides full-time employees with 22 working days of paid annual leave per leave year. Part-time employees receive a pro-rata entitlement based on their contracted hours.


In [7]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

# Configure LLM, Embedding, and Chunk Size
Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0.2)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")
Settings.chunk_size = 600
Settings.chunk_overlap = 200

# Define a system prompt
system_prompt = '''
You are Aisha, the Enterprise HR chatbot working for the DDS HR project. Your objective is to answer user questions strictly based on the four specific DDS HR policy documents provided.

Always follow these rules:
- ONLY use information from the four official DDS HR policy documents. If the answer is not directly covered, do not improvise or use internet knowledge.
- If a question relates to confidential information or cannot be answered based on the policy documents, politely instruct the user to send an email to connect@decodingdatascience.com for further assistance.
- If a question is unclear, ambiguous, or confusing, respond politely and ask the user to clarify or rephrase.
- Always be polite, professional, and concise in your responses.
- Never answer questions outside the scope of the DDS HR policy documents.

When answering:
- First, carefully check if the question matches information in the DDS HR policies.
- If relevant information is available, cite it directly and answer to the point.
- If the information is NOT present or the question is outside the scope, inform the user politely and guide them to email connect@decodingdatascience.com.
- If the question is unclear, gently request clarification.

Output Format:
- All responses should be short, professional, and to the point (typically 1-4 sentences).
- If unavailable or confidential, use the template: "I'm sorry, I can only answer questions from the official DDS HR policy documents. For anything further, please email connect@decodingdatascience.com."
- If clarification is needed, use the template: "Could you please clarify or rephrase your question regarding the DDS HR policy?"

Examples:

**Example 1**
User input: What is the leave policy for full-time employees?
Aisha output: According to the DDS HR policy, full-time employees are entitled to [X] days of paid leave per year. Please refer to section [Y] of the policy for full details.

**Example 2**
User input: What is the company’s stock price today?
Aisha output: I'm sorry, I can only answer questions from the official DDS HR policy documents. For anything further, please email connect@decodingdatascience.com.

**Example 3**
User input: How do I request remote work under the DDS HR guidelines?
Aisha output: As per DDS HR policies, employees may request remote work by [specific process—placeholder]. Please see section [Z] of the DDS HR policy document for more information.

**Example 4 (unclear question)**
User input: Benefits?
Aisha output: Could you please clarify or rephrase your question regarding the DDS HR policy?

(For real examples, replace [X], [Y], [Z], and [specific process—placeholder] with actual document content as available.)

---
Key Instructions Reminder:
Only answer using the four DDS HR policy documents. If information is missing, confidential, or out of scope, refer the user to connect@decodingdatascience.com. Always be concise, polite, and professional. Request clarification if necessary. NEVER use external information.
'''



index = VectorStoreIndex.from_documents(documents=documents)

# Configure query engine with system prompt
query_engine = index.as_query_engine(system_prompt=system_prompt)

response = query_engine.query("What are decoding data science standard office hours in Dubai?")
print(response)

The standard office hours for Decoding Data Science (DDS) in Dubai are from 9:00 AM to 6:00 PM, Monday to Friday.


In [8]:
import gradio as gr
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

# Configure LLM, Embedding, and Chunk Size
Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0.2)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-large")
Settings.chunk_size = 600
Settings.chunk_overlap = 200

# Load data and build the index
documents = SimpleDirectoryReader("data").load_data()
index = VectorStoreIndex.from_documents(documents=documents)
query_engine = index.as_query_engine()

# Function to handle queries
def query_document(query):
    response = query_engine.query(query)
    return str(response)

# Gradio interface
interface = gr.Interface(
    fn=query_document,
    inputs=gr.Textbox(label="Enter your query", placeholder="Type your question here..."),
    outputs=gr.Textbox(label="Response"),
    title="DDS Enterise Chatbot connect to Data",
    description="Ask questions about the documents loaded into the system."
)

# Launch the Gradio app
if __name__ == "__main__":
    interface.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fb6660dc9e3dc02a54.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [9]:
#storing the vector store in local file
import os
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine

from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    StorageContext,
    load_index_from_storage,
)

# check if storage already exists
PERSIST_DIR = "./storage"
if not os.path.exists(PERSIST_DIR):
    # load the documents and create the index
    documents = SimpleDirectoryReader("data").load_data()
    index = VectorStoreIndex.from_documents(documents)
    # store it for later
    index.storage_context.persist(persist_dir=PERSIST_DIR)
else:
    # load the existing index
    storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
    index = load_index_from_storage(storage_context)

# Either way we can now query the index
query_engine = index.as_query_engine()

retriever = VectorIndexRetriever(index=index, similarity_top_k=3)

query_engine = RetrieverQueryEngine(retriever=retriever)

response = query_engine.query("What are DDS standard office hours in Dubai?")
print(response)


The standard office hours for DDS in Dubai are from 9:00 AM to 6:00 PM, Monday to Friday.


In [10]:
#timing
import os
import time
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine

from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    StorageContext,
    load_index_from_storage,
)

# Start timer for index setup
start_time = time.time()

# check if storage already exists
PERSIST_DIR = "./storage"
if not os.path.exists(PERSIST_DIR):
    # load the documents and create the index
    documents = SimpleDirectoryReader("data").load_data()
    index = VectorStoreIndex.from_documents(documents)
    # store it for later
    index.storage_context.persist(persist_dir=PERSIST_DIR)
else:
    # load the existing index
    storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
    index = load_index_from_storage(storage_context)

setup_duration = time.time() - start_time
print(f"Index setup time: {setup_duration:.2f} seconds")

# Start timer for query
query_start_time = time.time()

# Prepare the query engine
retriever = VectorIndexRetriever(index=index, similarity_top_k=2)
query_engine = RetrieverQueryEngine(retriever=retriever)

# Execute query
response = query_engine.query("Who all mentioned in the doc?")
print(response)

query_duration = time.time() - query_start_time
print(f"Query time: {query_duration:.2f} seconds")


Index setup time: 0.36 seconds
The document does not mention specific individuals by name. It refers to roles such as managers, HR/People Partners, and leadership, but does not provide any personal names or specific individuals.
Query time: 1.84 seconds


In [13]:
import os
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine

from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    StorageContext,
    load_index_from_storage,
)

import time
start_time = time.time()
query_engine = index.as_query_engine()

retriever = VectorIndexRetriever(index=index, similarity_top_k=2)

query_engine = RetrieverQueryEngine(retriever=retriever)

response = query_engine.query("What are DDS standard office hours in Dubai??")
print(response)


end_time = time.time()  # Record end time
execution_time = end_time - start_time  # Calculate execution time
print(f"Execution time: {execution_time} seconds")

The standard office hours for DDS in Dubai are from 9:00 AM to 6:00 PM, Monday to Friday.
Execution time: 1.1588549613952637 seconds
